# 🚀 Rocket Engine Safety Shutdown System
### Collins Aerospace | AEGIS — Autonomous Engine Guardian & Intelligent Safety
**Framework:** LangGraph Reflex Agent + Groq LLM + Gradio UI

---
**Architecture:** Simple Reflex Agent
```
Sensor Input → [Evaluate Sensors] → [LLM Report Node] → Decorated Output → Gradio UI
                      ↓
         Temp > 3200°R → AUTO SHUTDOWN
         Temp > 2800°R → WARNING
         Temp ≤ 2800°R → NOMINAL
```
---

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Necessary Libraries                    ║
# ╚══════════════════════════════════════════════════════════╝
!pip install -q langgraph langchain langchain-groq gradio pydantic
print('✅ All libraries installed successfully!')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Import Libraries                               ║
# ╚══════════════════════════════════════════════════════════╝
import os
import time
import json
import random
import gradio as gr

from typing import TypedDict, Literal, Optional
from datetime import datetime

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

print('✅ All imports successful!')
print(f'   LangGraph  : ✓')
print(f'   LangChain  : ✓')
print(f'   Groq LLM   : ✓')
print(f'   Gradio     : ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Groq API Key Configuration                     ║
# ╚══════════════════════════════════════════════════════════╝
# Replace with your actual Groq API key from https://console.groq.com
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"   # <────── REPLACE THIS

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

if GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
    print('⚠️  WARNING: Please replace GROQ_API_KEY with your actual key!')
    print('   Get your free key at: https://console.groq.com')
else:
    print(f'✅ Groq API Key loaded: {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — LangGraph Reflex Agent: Safety Shutdown Logic  ║
# ╚══════════════════════════════════════════════════════════╝

# ── 4.1  Safety Thresholds (Rocket Engine Operating Limits) ─
TEMP_THRESHOLD_WARN  = 2800   # °R  — Yellow-zone: issue WARNING
TEMP_THRESHOLD_CRIT  = 3200   # °R  — Red-zone: trigger AUTO SHUTDOWN
PRESSURE_MAX         = 3600   # psi — Max allowable chamber pressure
VIBRATION_MAX        = 18.5   # g   — Max structural vibration limit

print(f'🔒 Safety Thresholds Loaded:')
print(f'   Temp Warning  : {TEMP_THRESHOLD_WARN:,} °R')
print(f'   Temp Critical : {TEMP_THRESHOLD_CRIT:,} °R  → AUTO SHUTDOWN')
print(f'   Max Pressure  : {PRESSURE_MAX:,} psi')
print(f'   Max Vibration : {VIBRATION_MAX} g')

# ── 4.2  Agent State Schema (LangGraph TypedDict) ────────────
class EngineState(TypedDict):
    engine_id:          str
    temperature:        float    # Rankine — engine combustion temp
    chamber_pressure:   float    # psi     — combustion chamber pressure
    vibration_g:        float    # g-force — structural vibration
    timestamp:          str      # ISO 8601 UTC
    decision:           str      # 'NOMINAL' | 'WARNING' | 'SHUTDOWN'
    llm_report:         str      # AEGIS LLM-generated report
    shutdown_triggered: bool     # True if automatic shutdown engaged


# ── 4.3  Node 1: Pure-Logic Sensor Evaluation (Reflex Rule) ──
def evaluate_sensors(state: EngineState) -> EngineState:
    """
    REFLEX RULE:
      IF engine_temp > TEMP_THRESHOLD_CRIT  →  SHUTDOWN
      IF engine_temp > TEMP_THRESHOLD_WARN  →  WARNING
      ELSE                                  →  NOMINAL
    Also checks pressure and vibration for compound failure.
    """
    temp = state['temperature']
    pres = state['chamber_pressure']
    vib  = state['vibration_g']

    if temp > TEMP_THRESHOLD_CRIT or pres > PRESSURE_MAX or vib > VIBRATION_MAX:
        state['decision']           = 'SHUTDOWN'
        state['shutdown_triggered'] = True
    elif temp > TEMP_THRESHOLD_WARN:
        state['decision']           = 'WARNING'
        state['shutdown_triggered'] = False
    else:
        state['decision']           = 'NOMINAL'
        state['shutdown_triggered'] = False

    print(f'  [SENSOR NODE] Engine {state["engine_id"]} → Decision: {state["decision"]}')
    return state


# ── 4.4  Node 2: LLM Report Generator ────────────────────────
def generate_llm_report(state: EngineState) -> EngineState:
    """
    Calls the open-source LLM via Groq API.
    Model: llama-3.3-70b-versatile
    (Maps to gpt-oss-120b endpoint on your custom server —
     swap model name to your server's model string if self-hosted)
    """
    llm = ChatGroq(
        model='llama-3.3-70b-versatile',  # Change to your server model string
        temperature=0.3,
        max_tokens=700,
    )

    system_prompt = """You are AEGIS — the Autonomous Engine Guardian & Intelligent Safety system
for Collins Aerospace Propulsion Division. You produce concise, authoritative,
engineering-grade safety incident reports.

FORMAT YOUR RESPONSE EXACTLY AS SHOWN BELOW (preserve all lines and symbols):

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STATUS   : <🔴 SHUTDOWN ENGAGED / ⚠️ WARNING ACTIVE / ✅ NOMINAL>
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 TELEMETRY SUMMARY
  Engine ID            : <id>
  Timestamp            : <timestamp>
  Chamber Temperature  : <temp> °R        [Limit: 3,200 °R]
  Chamber Pressure     : <pressure> psi   [Limit: 3,600 psi]
  Structural Vibration : <vibration> g    [Limit: 18.5 g]
  Parameters Exceeded  : <list exceeded params or 'None'>

🔍 FAULT ANALYSIS
  <2–3 sentences of concise engineering root-cause analysis>

⚡ AUTOMATED ACTIONS EXECUTED
  • <action 1 — specific and technical>
  • <action 2>
  • <action 3>

📡 RECOMMENDED ENGINEER ACTIONS
  1. <specific step>
  2. <specific step>
  3. <specific step>

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  AEGIS v2.4 | Collins Aerospace | Propulsion Safety Division
  Report Classification: SAFETY-CRITICAL — Retain 10 years
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"""

    user_prompt = f"""Engine telemetry received for AEGIS evaluation:

  Engine ID           : {state['engine_id']}
  Timestamp           : {state['timestamp']}
  Temperature         : {state['temperature']:.1f} °R   (Critical limit: {TEMP_THRESHOLD_CRIT} °R)
  Chamber Pressure    : {state['chamber_pressure']:.1f} psi  (Max limit: {PRESSURE_MAX} psi)
  Vibration Level     : {state['vibration_g']:.2f} g     (Max limit: {VIBRATION_MAX} g)
  AEGIS Reflex Result : {state['decision']}
  Shutdown Triggered  : {state['shutdown_triggered']}

Generate the full AEGIS safety incident report."""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]

    print(f'  [LLM NODE] Generating AEGIS report via Groq...')
    response = llm.invoke(messages)
    state['llm_report'] = response.content
    print(f'  [LLM NODE] Report generated ({len(response.content)} chars)')
    return state


# ── 4.5  Conditional Router ───────────────────────────────────
def route_decision(state: EngineState) -> Literal['generate_report', '__end__']:
    # Always generate a report for all decisions (NOMINAL, WARNING, SHUTDOWN)
    return 'generate_report'


# ── 4.6  Assemble the LangGraph ───────────────────────────────
def build_agent() -> StateGraph:
    graph = StateGraph(EngineState)

    # Register nodes
    graph.add_node('evaluate_sensors', evaluate_sensors)
    graph.add_node('generate_report',  generate_llm_report)

    # Entry point
    graph.set_entry_point('evaluate_sensors')

    # Conditional edge: after sensor evaluation → route
    graph.add_conditional_edges(
        'evaluate_sensors',
        route_decision,
        {'generate_report': 'generate_report'}
    )

    # Final edge: report → END
    graph.add_edge('generate_report', END)

    return graph.compile()


# Compile the agent
AGENT = build_agent()
print('\n🧠 LangGraph Reflex Agent compiled successfully!')
print('   Nodes: evaluate_sensors → generate_report → END')


# ── 4.7  Main Orchestrator Function ──────────────────────────
def run_safety_check(engine_id: str,
                     temperature: float,
                     chamber_pressure: float,
                     vibration_g: float) -> dict:
    """Invoke the AEGIS reflex agent with telemetry data."""
    initial_state: EngineState = {
        'engine_id':          engine_id,
        'temperature':        temperature,
        'chamber_pressure':   chamber_pressure,
        'vibration_g':        vibration_g,
        'timestamp':          datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        'decision':           '',
        'llm_report':         '',
        'shutdown_triggered': False,
    }
    print(f'\n🚀 AEGIS Invoked — Engine: {engine_id}')
    result = AGENT.invoke(initial_state)
    print(f'✅ Agent completed — Final Decision: {result["decision"]}')
    return result


print('\n✅ CELL 4 complete — Agent logic ready!')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Gradio Professional UI (Collins Aerospace)     ║
# ╚══════════════════════════════════════════════════════════╝

# ── CSS: Collins Aerospace Light Theme ───────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;600;800&family=Inter:wght@300;400;500;600&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --ca-navy:   #0A1628;
    --ca-blue:   #003087;
    --ca-sky:    #00A3E0;
    --ca-ice:    #E8F4FD;
    --ca-silver: #CDD6E0;
    --ca-white:  #FFFFFF;
    --ca-warn:   #F59E0B;
    --ca-danger: #DC2626;
    --ca-safe:   #16A34A;
    --ca-panel:  #F0F6FB;
    --ca-border: #B8D0E8;
    --shadow:    0 4px 24px rgba(0,48,135,0.10);
}

* { box-sizing: border-box; }

body, .gradio-container {
    background: linear-gradient(160deg, #EAF3FC 0%, #F7FBFF 60%, #E0EEF9 100%) !important;
    font-family: 'Inter', sans-serif !important;
    color: var(--ca-navy) !important;
}

/* ── Header ── */
.ca-header {
    background: linear-gradient(90deg, #0A1628 0%, #003087 60%, #004AAD 100%);
    border-radius: 0 0 18px 18px;
    overflow: hidden;
    box-shadow: 0 6px 32px rgba(0,48,135,0.22);
    margin-bottom: 24px;
}
.ca-header-inner {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 22px 40px 16px 40px;
    flex-wrap: wrap;
    gap: 12px;
}
.ca-brand-name {
    font-family: 'Orbitron', monospace;
    font-weight: 800;
    font-size: 20px;
    color: #ffffff;
    letter-spacing: 1.5px;
}
.ca-brand-sub {
    font-size: 10px;
    color: #00A3E0;
    letter-spacing: 3px;
    text-transform: uppercase;
    margin-top: 3px;
}
.ca-title-center {
    text-align: center;
    flex: 1;
}
.ca-title-main {
    font-family: 'Orbitron', monospace;
    font-size: 16px;
    font-weight: 600;
    color: #ffffff;
    letter-spacing: 2px;
}
.ca-title-sub {
    font-size: 10px;
    color: #CDD6E0;
    letter-spacing: 2px;
    margin-top: 4px;
}
.ca-live-badge {
    background: rgba(0,163,224,0.18);
    border: 1px solid #00A3E0;
    border-radius: 20px;
    padding: 6px 16px;
    font-size: 10px;
    color: #00A3E0;
    font-weight: 700;
    letter-spacing: 2px;
    font-family: 'JetBrains Mono', monospace;
}

/* ── Moving Scanner Line ── */
.ca-scanner {
    width: 100%;
    height: 4px;
    background: rgba(255,255,255,0.07);
    overflow: hidden;
    position: relative;
}
.ca-scanner-beam {
    height: 4px;
    width: 20%;
    background: linear-gradient(90deg, transparent, #00A3E0, #7FD4F5, #00A3E0, transparent);
    position: absolute;
    top: 0; left: -22%;
    animation: beam 2.6s ease-in-out infinite;
}
@keyframes beam {
    0%   { left: -22%; opacity: 0.5; }
    50%  { opacity: 1; }
    100% { left: 106%; opacity: 0.5; }
}

/* ── Metric Cards ── */
.metric-row {
    display: flex;
    gap: 10px;
    margin-bottom: 20px;
    flex-wrap: wrap;
}
.metric-card {
    flex: 1;
    min-width: 100px;
    background: #F0F6FB;
    border: 1px solid #B8D0E8;
    border-radius: 8px;
    padding: 12px 10px;
    text-align: center;
}
.metric-label {
    font-size: 8px;
    color: #6A8FAF;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin-bottom: 5px;
    font-weight: 600;
}
.metric-value {
    font-family: 'JetBrains Mono', monospace;
    font-size: 17px;
    font-weight: 600;
    color: #003087;
}
.metric-unit {
    font-size: 9px;
    color: #7A9AB8;
    margin-top: 2px;
}

/* ── Section titles ── */
.sec-title {
    font-family: 'Orbitron', monospace;
    font-size: 10px;
    font-weight: 600;
    color: #00A3E0;
    letter-spacing: 2.5px;
    text-transform: uppercase;
    margin-bottom: 14px;
    padding-bottom: 8px;
    border-bottom: 1px solid #D4E8F5;
}

/* ── Inputs ── */
label > span {
    font-family: 'Inter', sans-serif !important;
    font-size: 11px !important;
    font-weight: 600 !important;
    color: #4A6FA5 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.8px !important;
}
input[type=range] { accent-color: #00A3E0 !important; }
input[type=text], input[type=number], textarea {
    background: #EAF3FC !important;
    border: 1.5px solid #B8D0E8 !important;
    border-radius: 7px !important;
    color: #0A1628 !important;
    font-family: 'JetBrains Mono', monospace !important;
}

/* ── Run Button ── */
button.ca-run {
    background: linear-gradient(135deg, #003087 0%, #00A3E0 100%) !important;
    color: white !important;
    font-family: 'Orbitron', monospace !important;
    font-weight: 700 !important;
    font-size: 12px !important;
    letter-spacing: 2.5px !important;
    padding: 14px !important;
    border-radius: 8px !important;
    border: none !important;
    box-shadow: 0 4px 18px rgba(0,48,135,0.30) !important;
    transition: transform .15s, box-shadow .15s !important;
    width: 100% !important;
}
button.ca-run:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 26px rgba(0,163,224,0.38) !important;
}

/* ── Sim Buttons ── */
button.ca-sim {
    background: #1E3A5F !important;
    color: #A8D8F0 !important;
    font-family: 'Orbitron', monospace !important;
    font-size: 9px !important;
    letter-spacing: 1px !important;
    padding: 10px 6px !important;
    border-radius: 7px !important;
    border: 1px solid #2D6A9F !important;
    width: 100% !important;
}

/* ── Output terminal ── */
.ca-out textarea {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12.5px !important;
    line-height: 1.75 !important;
    background: #071020 !important;
    color: #A8D8F0 !important;
    border: 1.5px solid #1A3A5C !important;
    border-radius: 10px !important;
    padding: 18px !important;
    min-height: 460px !important;
}

/* ── Footer ── */
.ca-footer {
    text-align: center;
    padding: 18px;
    font-size: 9px;
    color: #7A9AB8;
    letter-spacing: 2px;
    text-transform: uppercase;
    border-top: 1px solid #D0E4F0;
    margin-top: 20px;
    font-family: 'JetBrains Mono', monospace;
}

.gradio-container { max-width: 1200px !important; margin: 0 auto !important; padding: 0 16px !important; }
"""

HEADER_HTML = """
<div class="ca-header">
  <div class="ca-header-inner">
    <div>
      <div class="ca-brand-name">Collins Aerospace</div>
      <div class="ca-brand-sub">Propulsion Safety Systems</div>
    </div>
    <div class="ca-title-center">
      <div class="ca-title-main">🚀 ROCKET ENGINE SAFETY SHUTDOWN</div>
      <div class="ca-title-sub">AEGIS — Autonomous Engine Guardian &amp; Intelligent Safety System</div>
    </div>
    <div class="ca-live-badge">⬤ LIVE</div>
  </div>
  <div class="ca-scanner">
    <div class="ca-scanner-beam"></div>
  </div>
</div>
"""

THRESHOLDS_HTML = """
<div class="metric-row">
  <div class="metric-card">
    <div class="metric-label">Temp Warning</div>
    <div class="metric-value" style="color:#D97706">2,800</div>
    <div class="metric-unit">°R</div>
  </div>
  <div class="metric-card">
    <div class="metric-label">Temp Critical</div>
    <div class="metric-value" style="color:#DC2626">3,200</div>
    <div class="metric-unit">°R — Shutdown</div>
  </div>
  <div class="metric-card">
    <div class="metric-label">Max Pressure</div>
    <div class="metric-value">3,600</div>
    <div class="metric-unit">psi</div>
  </div>
  <div class="metric-card">
    <div class="metric-label">Max Vibration</div>
    <div class="metric-value">18.5</div>
    <div class="metric-unit">g</div>
  </div>
</div>
"""


# ── Simulation scenario helpers ───────────────────────────────
def scenario_nominal():
    return 'ENG-001', round(random.uniform(2100, 2750), 1), round(random.uniform(2500, 3200), 1), round(random.uniform(3.0, 13.0), 2)

def scenario_warning():
    return 'ENG-002', round(random.uniform(2801, 3150), 1), round(random.uniform(3000, 3500), 1), round(random.uniform(12.0, 17.5), 2)

def scenario_shutdown():
    return 'ENG-003', round(random.uniform(3201, 4000), 1), round(random.uniform(3601, 4400), 1), round(random.uniform(18.6, 28.0), 2)


# ── Gradio callback ───────────────────────────────────────────
def run_check(engine_id, temperature, pressure, vibration):
    if not str(engine_id).strip():
        engine_id = 'ENG-001'
    try:
        result = run_safety_check(
            str(engine_id).strip().upper(),
            float(temperature),
            float(pressure),
            float(vibration)
        )
        decision = result['decision']
        shutdown = result['shutdown_triggered']

        header  = f"\n{'═'*58}\n"
        header += f"  ENGINE   : {result['engine_id']}\n"
        header += f"  TIME     : {result['timestamp']}\n"
        header += f"  DECISION : {decision}"
        if shutdown:
            header += "   ← 🔴 AUTOMATIC SHUTDOWN ENGAGED"
        elif decision == 'WARNING':
            header += "   ← ⚠️ WARNING ACTIVE"
        else:
            header += "   ← ✅ SYSTEM NOMINAL"
        header += f"\n{'═'*58}\n\n"

        return header + result['llm_report']
    except Exception as e:
        return (
            f"\n{'═'*58}\n"
            f"  ⚠️  AGENT ERROR\n"
            f"{'═'*58}\n\n"
            f"  {str(e)}\n\n"
            f"  Check: GROQ_API_KEY is valid and network is reachable.\n"
            f"  Groq Console: https://console.groq.com\n"
        )


# ── Build Gradio App ──────────────────────────────────────────
with gr.Blocks(css=CSS, title='AEGIS | Rocket Engine Safety | Collins Aerospace') as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row():
        # ── LEFT: Input Panel ──
        with gr.Column(scale=1, min_width=320):
            gr.HTML('<div class="sec-title">⚙️ Engine Telemetry Input</div>')
            gr.HTML(THRESHOLDS_HTML)

            engine_id = gr.Textbox(
                label='Engine Identification',
                value='ENG-001',
                placeholder='e.g. ENG-001'
            )
            temperature = gr.Slider(
                label='🌡️ Temperature (°R)',
                minimum=1000, maximum=4500, value=2500, step=10,
                info='Warning >2,800 | Shutdown >3,200 °R'
            )
            chamber_pressure = gr.Slider(
                label='🔩 Chamber Pressure (psi)',
                minimum=500, maximum=5000, value=3000, step=10,
                info='Structural limit: 3,600 psi'
            )
            vibration = gr.Slider(
                label='📳 Vibration (g)',
                minimum=0.0, maximum=30.0, value=8.0, step=0.1,
                info='Structural limit: 18.5 g'
            )

            btn_run = gr.Button(
                '▶  RUN SAFETY CHECK',
                elem_classes=['ca-run']
            )

            gr.HTML('<div class="sec-title" style="margin-top:22px">🧪 Simulation Scenarios</div>')
            with gr.Row():
                btn_nom = gr.Button('✅ NOMINAL',  elem_classes=['ca-sim'])
                btn_war = gr.Button('⚠️ WARNING',  elem_classes=['ca-sim'])
                btn_shd = gr.Button('🔴 SHUTDOWN', elem_classes=['ca-sim'])

        # ── RIGHT: AEGIS Report Output ──
        with gr.Column(scale=2):
            gr.HTML('<div class="sec-title">📡 AEGIS Safety Incident Report</div>')
            output = gr.Textbox(
                label='',
                lines=30,
                interactive=False,
                placeholder=(
                    '\n'
                    '  ══════════════════════════════════════════════════════\n'
                    '  AEGIS  —  Autonomous Engine Guardian & Safety System  \n'
                    '  Collins Aerospace | Propulsion Safety Division        \n'
                    '  ══════════════════════════════════════════════════════\n\n'
                    '  System standing by.\n\n'
                    '  Configure engine telemetry parameters in the left     \n'
                    '  panel and press  ▶ RUN SAFETY CHECK  to invoke the   \n'
                    '  AEGIS reflex agent.\n\n'
                    '  Or load a simulation scenario using the buttons below:\n'
                    '    ✅ NOMINAL   — All parameters within safe range\n'
                    '    ⚠️ WARNING   — Temperature approaching critical\n'
                    '    🔴 SHUTDOWN  — Critical threshold exceeded\n\n'
                    '  ──────────────────────────────────────────────────────\n'
                    '  AEGIS v2.4  |  LangGraph Reflex Agent  |  Groq LLM\n'
                ),
                elem_classes=['ca-out']
            )

    gr.HTML("""
    <div class="ca-footer">
      Collins Aerospace &nbsp;|&nbsp; AEGIS Propulsion Safety Division &nbsp;|&nbsp;
      LangGraph Reflex Agent Framework &nbsp;|&nbsp;
      Classification: SAFETY-CRITICAL — INTERNAL USE ONLY
    </div>
    """)

    # ── Wire Events ──────────────────────────────────────────
    btn_run.click(
        fn=run_check,
        inputs=[engine_id, temperature, chamber_pressure, vibration],
        outputs=output
    )

    btn_nom.click(fn=scenario_nominal, outputs=[engine_id, temperature, chamber_pressure, vibration])
    btn_war.click(fn=scenario_warning, outputs=[engine_id, temperature, chamber_pressure, vibration])
    btn_shd.click(fn=scenario_shutdown, outputs=[engine_id, temperature, chamber_pressure, vibration])

print('✅ Gradio UI built successfully!')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Launch the Application                         ║
# ╚══════════════════════════════════════════════════════════╝
print('🚀 Launching AEGIS — Rocket Engine Safety Shutdown System...')
print('   Collins Aerospace | Propulsion Safety Division')
print('   A public share link will appear below.\n')

demo.launch(
    share=True,      # Creates public URL for Colab
    debug=False,     # Set True to see LangGraph trace logs
    quiet=False
)